# Experiment 4 — Transformer

In [3]:
import pandas as pd

df_clean = pd.read_csv("../data/cleaned_data.csv")

df_clean["date"] = pd.to_datetime(df_clean["date"])
df_clean = df_clean.iloc[:, 1:3]
df_clean.head()

,date,demand
0,2015-01-01,99635.030
1,2015-01-02,129606.010
2,2015-01-03,142300.540
3,2015-01-04,104330.715
4,2015-01-05,118132.200


In [4]:
import torch
import torch.nn as nn

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

## Chronological Train/test split

In [5]:
import numpy as np

# train-test split

df = df_clean.set_index("date")
df = df.sort_index()

# demand is the only feature
values = df["demand"].values

n = len(values)

train_end = int(0.8 * n)

train_values = values[:train_end]
test_values = values[train_end:]

print(f"Train: {len(train_values)}")
print(f"Test: {len(test_values)}")

Train: 1684
Test: 422


## Data Transformation

In [6]:
# min-max scaling (fit on train only, applied to both)
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(
    train_values.reshape(-1, 1)
)
test_scaled = scaler.transform(
    test_values.reshape(-1, 1)
)
print(np.shape(train_scaled))
print(np.shape(test_scaled))

(1684, 1)
(422, 1)


## Create the sliding window sequences

Use the previous `sequence_length` days of demand to predict the next day's demand.

In [7]:
SEQUENCE_LENGTH = 7


def create_sequences(values: np.ndarray, sequence_length: int):
    """
    Create sliding-window sequences for next-day forecasting.

    Example with sequence_length=7:

        X = days 1-7
        y = day 8

        X = days 2-8
        y = day 9
    """

    X = []
    y = []

    for i in range(len(values) - sequence_length):
        X.append(values[i : i + sequence_length])
        y.append(values[i + sequence_length])

    return np.array(X), np.array(y)


X_train, y_train = create_sequences(train_scaled, SEQUENCE_LENGTH)
X_test, y_test = create_sequences(test_scaled, SEQUENCE_LENGTH)

print(np.shape(X_train))
print(np.shape(y_train))

(1677, 7, 1)
(1677, 1)


## Convert to PyTorch Tensors

In [8]:
# Transformer encoder expects: (batch_size, sequence_length, input_size)
# create_sequences already preserves the feature dim from the (N, 1) scaled arrays,
# so X is already (N, seq_len, 1) and y is already (N, 1).
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

torch.Size([1677, 7, 1])
torch.Size([1677, 1])
torch.Size([415, 7, 1])
torch.Size([415, 1])


## Data Loader

In [9]:
from torch.utils.data import DataLoader, TensorDataset

BATCH_SIZE = 64

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## Create the model

Uses PyTorch's built-in `nn.TransformerEncoder` (vs. the recurrent models in Experiments 1-3). Unlike an RNN/LSTM/GRU, self-attention processes all timesteps in parallel with no inherent sense of order, so we add a sinusoidal **positional encoding** to the input embeddings before the encoder. Many-to-one: since forecasting isn't autoregressive here (we're not generating tokens step-by-step), no causal mask is needed — the encoder attends over the full input window, and the representation at the final timestep is fed to a linear output layer to make one prediction per sequence.

In [10]:
import math


class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding, added to input embeddings."""

    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))

        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # buffer, not a parameter: moves with .to(device) but isn't learned/saved as trainable
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch_size, sequence_length, d_model)
        return x + self.pe[:, : x.size(1), :]


class ElectricityDemandTransformer(nn.Module):
    """Transformer encoder built on PyTorch's built-in nn.TransformerEncoder."""

    def __init__(self, input_size: int, d_model: int, output_size: int,
                 nhead: int = 4, num_layers: int = 2, dim_feedforward: int = 64,
                 dropout: float = 0.1):
        """
        Args:
            input_size: How many features come into the model at one timestep.
            d_model: Internal embedding dimension the transformer operates in.
            output_size: How many values you want the model to predict.
            nhead: Number of self-attention heads (must divide d_model).
            num_layers: How many stacked encoder layers.
            dim_feedforward: Hidden size of the per-layer feed-forward network.
            dropout: Dropout probability inside the encoder layers.
        """
        super().__init__()

        self.d_model = d_model

        # project the single demand feature into the model's embedding dimension
        self.input_proj = nn.Linear(input_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.fc = nn.Linear(d_model, output_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input sequence with shape (batch_size, sequence_length, input_size).

        Returns:
            One prediction per sequence with shape (batch_size, output_size).
        """

        x = self.input_proj(x) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)

        # encoder_out: (batch_size, sequence_length, d_model)
        encoder_out = self.encoder(x)

        # many-to-one: use the representation at the final timestep
        last_hidden = encoder_out[:, -1, :]

        return self.fc(last_hidden)

In [11]:
INPUT_SIZE = train_scaled.shape[-1]   # 1
OUTPUT_SIZE = train_scaled.shape[-1]  # 1
D_MODEL = 32
NHEAD = 4
NUM_LAYERS = 2
DIM_FEEDFORWARD = 64
DROPOUT = 0.1

EPOCHS = 10

model = ElectricityDemandTransformer(
    INPUT_SIZE, D_MODEL, OUTPUT_SIZE,
    nhead=NHEAD, num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD, dropout=DROPOUT,
).to(device)
criterion = nn.L1Loss()  # pytorch's MAE
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))


def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


for name, param in model.named_parameters():
    print(name, param.shape)

input_proj.weight torch.Size([32, 1])
input_proj.bias torch.Size([32])
encoder.layers.0.self_attn.in_proj_weight torch.Size([96, 32])
encoder.layers.0.self_attn.in_proj_bias torch.Size([96])
encoder.layers.0.self_attn.out_proj.weight torch.Size([32, 32])
encoder.layers.0.self_attn.out_proj.bias torch.Size([32])
encoder.layers.0.linear1.weight torch.Size([64, 32])
encoder.layers.0.linear1.bias torch.Size([64])
encoder.layers.0.linear2.weight torch.Size([32, 64])
encoder.layers.0.linear2.bias torch.Size([32])
encoder.layers.0.norm1.weight torch.Size([32])
encoder.layers.0.norm1.bias torch.Size([32])
encoder.layers.0.norm2.weight torch.Size([32])
encoder.layers.0.norm2.bias torch.Size([32])
encoder.layers.1.self_attn.in_proj_weight torch.Size([96, 32])
encoder.layers.1.self_attn.in_proj_bias torch.Size([96])
encoder.layers.1.self_attn.out_proj.weight torch.Size([32, 32])
encoder.layers.1.self_attn.out_proj.bias torch.Size([32])
encoder.layers.1.linear1.weight torch.Size([64, 32])
encoder.

## Training Loop

In [12]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):

    model.train()

    total_loss = 0

    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        y_pred = model(x_batch)

        loss = criterion(y_pred, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    return avg_loss

## Train for multiple Epochs (single run, sequence_length=7)

In [13]:
for epoch in range(EPOCHS):

    train_loss = train_one_epoch(
        model=model,
        train_loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device
    )

    print(f"Epoch {epoch + 1}/{EPOCHS} | Loss: {train_loss:.4f}")

Epoch 1/10 | Loss: 0.1564
Epoch 2/10 | Loss: 0.1208
Epoch 3/10 | Loss: 0.1122
Epoch 4/10 | Loss: 0.0951
Epoch 5/10 | Loss: 0.0922
Epoch 6/10 | Loss: 0.0920
Epoch 7/10 | Loss: 0.0894
Epoch 8/10 | Loss: 0.0840
Epoch 9/10 | Loss: 0.0830
Epoch 10/10 | Loss: 0.0820


## Evaluate

Note: the training loss printed above is MAE on **min-max scaled** demand, not real units. Inverse-transform predictions back to the original demand scale before computing MAE/RMSE.

In [14]:
def evaluate_model(model, data_loader, scaler, device):
    """
    Run the model on a data loader and return predictions/targets
    in the ORIGINAL demand scale, plus real-unit MAE/RMSE.
    """

    model.eval()

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for x_batch, y_batch in data_loader:
            x_batch = x_batch.to(device)

            y_pred = model(x_batch)

            all_preds.append(y_pred.cpu())
            all_targets.append(y_batch.cpu())

    preds_scaled = torch.cat(all_preds).numpy()
    targets_scaled = torch.cat(all_targets).numpy()

    # undo min-max scaling to get back to real demand units
    preds = scaler.inverse_transform(preds_scaled)
    targets = scaler.inverse_transform(targets_scaled)

    test_mae = mae(targets, preds)
    test_rmse = rmse(targets, preds)

    return preds, targets, test_mae, test_rmse


preds, targets, test_mae, test_rmse = evaluate_model(model, test_loader, scaler, device)

print(f"Test MAE:  {test_mae:,.2f} demand units")
print(f"Test RMSE: {test_rmse:,.2f} demand units")

Test MAE:  6,575.72 demand units
Test RMSE: 8,551.78 demand units


## Sweep: sequence length vs. performance

Repeat the full pipeline (sequence creation → tensors → loader → fresh model → train → evaluate) for `sequence_length` in `[1, 7, 14, 30, 60]`, and record MAE, RMSE, training time, and inference latency for each — same methodology as Experiments 1-3 (vanilla RNN, LSTM, GRU).

In [15]:
import time


def run_experiment(sequence_length: int, d_model: int = D_MODEL, nhead: int = NHEAD,
                    num_layers: int = NUM_LAYERS, dim_feedforward: int = DIM_FEEDFORWARD,
                    dropout: float = DROPOUT, epochs: int = EPOCHS,
                    batch_size: int = BATCH_SIZE, lr: float = 0.001, seed: int = 42):
    """
    Build sequences, train a fresh ElectricityDemandTransformer, and evaluate it
    for a given sequence_length. Returns a dict of results.

    train_scaled / test_scaled / scaler / device are reused from the
    outer scope (already fit on the chronological train/test split).
    """

    torch.manual_seed(seed)

    # 1. sliding-window sequences for this sequence_length
    X_tr, y_tr = create_sequences(train_scaled, sequence_length)
    X_te, y_te = create_sequences(test_scaled, sequence_length)

    # 2. tensors — already (N, seq_len, 1) / (N, 1), no unsqueeze needed
    X_tr = torch.tensor(X_tr, dtype=torch.float32)
    y_tr = torch.tensor(y_tr, dtype=torch.float32)
    X_te = torch.tensor(X_te, dtype=torch.float32)
    y_te = torch.tensor(y_te, dtype=torch.float32)

    # 3. loaders
    train_loader_exp = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=False)
    test_loader_exp = DataLoader(TensorDataset(X_te, y_te), batch_size=batch_size, shuffle=False)

    # 4. fresh model per run so results aren't contaminated by earlier training
    model_exp = ElectricityDemandTransformer(
        INPUT_SIZE, d_model, OUTPUT_SIZE,
        nhead=nhead, num_layers=num_layers,
        dim_feedforward=dim_feedforward, dropout=dropout,
    ).to(device)
    criterion_exp = nn.L1Loss()
    optimizer_exp = torch.optim.Adam(model_exp.parameters(), lr=lr)

    # 5. train, timing the whole training run
    train_start = time.perf_counter()
    for _ in range(epochs):
        train_one_epoch(
            model=model_exp,
            train_loader=train_loader_exp,
            criterion=criterion_exp,
            optimizer=optimizer_exp,
            device=device,
        )
    train_time = time.perf_counter() - train_start

    # 6. evaluate + inference latency (ms per sample, single-sample batches)
    single_loader = DataLoader(TensorDataset(X_te, y_te), batch_size=1, shuffle=False)
    model_exp.eval()
    with torch.no_grad():
        infer_start = time.perf_counter()
        for x_batch, _ in single_loader:
            model_exp(x_batch.to(device))
        infer_time = time.perf_counter() - infer_start
    latency_ms = (infer_time / len(single_loader.dataset)) * 1000

    _, _, test_mae, test_rmse = evaluate_model(model_exp, test_loader_exp, scaler, device)

    return {
        "sequence_length": sequence_length,
        "mae": test_mae,
        "rmse": test_rmse,
        "training_time_s": train_time,
        "inference_latency_ms": latency_ms,
    }

In [16]:
SEQUENCE_LENGTHS = [1, 7, 14, 30, 60]

results = [run_experiment(seq_len) for seq_len in SEQUENCE_LENGTHS]

results_df = pd.DataFrame(results).set_index("sequence_length")
results_df

,mae,rmse,training_time_s,inference_latency_ms
sequence_length,,,,
1,7516.348633,9807.522461,6.653051,2.551296
7,6575.718262,8551.777344,9.235628,3.075652
14,6325.072266,8458.743164,9.860706,4.038359
30,6783.849609,8927.573242,9.805944,3.678054
60,6921.336914,9088.635742,8.676324,2.256606
